In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
qs = pd.read_csv('/content/drive/MyDrive/Higher Education Performance/qs-world-rankings-2025.csv')
the = pd.read_csv('/content/drive/MyDrive/Higher Education Performance/THE World University Rankings 2016-2026.csv')

qs.info()
qs.describe()
print(qs.head())

the.info()
the.describe()
print(the.head())

qs = qs.drop_duplicates()
the = the.drop_duplicates()

qs.isnull().sum()
the.isnull().sum()

qs['Institution Name'] = qs['Institution Name'].str.strip().str.lower()
the['Name'] = the['Name'].str.strip().str.lower()
qs['Location'] = qs['Location'].str.strip().str.lower()
the['Country'] = the['Country'].str.strip().str.lower()

merged = pd.merge(qs, the, left_on=['Institution Name', 'Location'], right_on=['Name', 'Country'], how='outer')
merged.to_csv('/content/drive/MyDrive/university_cleaned.csv', index=False)
df = pd.read_csv('/content/drive/MyDrive/university_cleaned.csv')
# Clean 'Student Population' (from THE rankings in df) and 'International Students_y' (from THE rankings in df)
def clean_population_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace(',', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    elif '+' in s:
        try:
            return float(s.replace('+', ''))
        except ValueError:
            return None
    elif '~' in s:
        try:
            return float(s.replace('~', ''))
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

def clean_percentage_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace('%', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

df['Student_Population_Cleaned'] = df['Student Population'].apply(clean_population_range)
df['International_Students_y_Cleaned'] = df['International Students_y'].apply(clean_percentage_range)

df['Global_Rank_Score'] = 100 - df['Rank']   # Using 'Rank' from THE
df['Research_Productivity_Index'] = df['Research Quality'] / df['Students to Staff Ratio']
df['Faculty_Student_Ratio'] = df['Faculty Student']
df['Intl_Student_Percentage'] = df['International_Students_y_Cleaned']
df[['Student_Population_Cleaned','International_Students_y_Cleaned','Global_Rank_Score','Research_Productivity_Index']].describe()

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']] = scaler.fit_transform(
    df[['Global_Rank_Score','Research_Productivity_Index','Intl_Student_Percentage']]
)

df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)

df['Performance_Index'] = (
    0.4*df['Global_Rank_Score'] +
    0.3*df['Research_Productivity_Index'] +
    0.3*df['Intl_Student_Percentage']
)
import plotly.express as px

fig = px.bar(df.sort_values('Performance_Index',ascending=False).head(10),
             x='Institution Name', y='Performance_Index',
             title='Top 10 Universities by Performance Index')
fig.show()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   2025 Rank                       1503 non-null   object 
 1   2024 Rank                       1482 non-null   object 
 2   Institution Name                1503 non-null   object 
 3   Location                        1503 non-null   object 
 4   Location Full                   1503 non-null   object 
 5   Size                            1503 non-null   object 
 6   Academic Reputation             1503 non-null   float64
 7   Employer Reputation             1503 non-null   float64
 8   Faculty Student                 1503 non-null   float64
 9   Citations per Faculty           1503 non-null   float64
 10  International Faculty      

In [ ]:
df['University_Name'] = df['Institution Name'].fillna(df['Name'])

df['Country_Final'] = df['Location'].fillna(df['Country'])

df['Publications'] = df['Research Environment']

df['Citations'] = df['Research Quality']

region_map = {
    'india':'Asia',
    'china':'Asia',
    'japan':'Asia',
    'south korea':'Asia',
    'singapore':'Asia',
    'malaysia':'Asia',
    'thailand':'Asia',
    'united states':'North America',
    'canada':'North America',
    'mexico':'North America',
    'united kingdom':'Europe',
    'england':'Europe',
    'france':'Europe',
    'germany':'Europe',
    'italy':'Europe',
    'spain':'Europe',
    'netherlands':'Europe',
    'switzerland':'Europe',
    'sweden':'Europe',
    'norway':'Europe',
    'finland':'Europe',
    'denmark':'Europe',
    'ireland':'Europe',
    'belgium':'Europe',
    'australia':'Oceania',
    'new zealand':'Oceania',
    'brazil':'South America',
    'argentina':'South America',
    'chile':'South America',
    'south africa':'Africa',
    'egypt':'Africa',
    'kenya':'Africa'
}

df['Region'] = df['Country_Final'].map(region_map)
df['Region'] = df['Region'].fillna('Other')

df['Student_Diversity'] = df['Intl_Student_Percentage']

df.to_csv("/content/drive/MyDrive/university_final_dataset.csv", index=False)